In [3]:
import nbformat as nbf

# Créer un nouveau notebook
nb = nbf.v4.new_notebook()

# 1. Ajout des cellules pour l'EDA (Exploration des Données)
eda_cells = [
    nbf.v4.new_markdown_cell("""
# Exploration et Démonstration du Projet Titanic

Ce notebook combine l'analyse exploratoire des données (EDA) et l'exécution complète du pipeline de modélisation.
"""),
    nbf.v4.new_code_cell("""
# Importation des bibliothèques nécessaires
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys

current_dir = os.path.dirname(__file__)
project_root = os.path.abspath(os.path.join(current_dir, "../../"))
if project_root not in sys.path:
    sys.path.append(project_root)

from titanic.infrastructure.cleaning import clean_data
from titanic.domain.pipeline import prepare_data
from titanic.domain.models import train_model, evaluate_model, save_model, load_model

%matplotlib inline
"""),
    nbf.v4.new_code_cell("""
# Chargement des données brutes
train_data = pd.read_csv('data/raw/train.csv', encoding='utf-8')
print('Aperçu des données brutes:')
print(train_data.head())
"""),
    nbf.v4.new_code_cell("""
# Statistiques descriptives
print(train_data.describe())
print(train_data.info())
"""),
    nbf.v4.new_code_cell("""
# Visualisation des valeurs manquantes
plt.figure(figsize=(10, 6))
sns.heatmap(train_data.isnull(), cbar=False, cmap='viridis')
plt.title('Valeurs manquantes dans le jeu de données')
plt.show()
"""),
    nbf.v4.new_code_cell("""
# Distribution des âges
plt.figure(figsize=(8, 6))
sns.histplot(train_data['Age'].dropna(), kde=True)
plt.title('Distribution des âges')
plt.show()
"""),
    nbf.v4.new_code_cell("""
# Analyse des survivants par sexe
plt.figure(figsize=(8, 6))
sns.countplot(x='Sex', hue='Survived', data=train_data)
plt.title('Répartition des survivants par sexe')
plt.show()
"""),
]

# 2. Ajout des cellules pour l'exécution du pipeline
pipeline_cells = [
    nbf.v4.new_code_cell("""
# Nettoyage des données
cleaned_data = clean_data(train_data)
print('Aperçu des données nettoyées:')
print(cleaned_data.head())
"""),
    nbf.v4.new_code_cell("""
# Préparation des données
X_train, X_test, y_train, y_test = prepare_data(cleaned_data, target_column='Survived')
print('Taille de X_train:', X_train.shape)
print('Taille de X_test:', X_test.shape)
"""),
    nbf.v4.new_code_cell("""
# Entraînement du modèle
model = train_model(X_train, y_train)
print('Modèle entraîné avec succès.')
"""),
    nbf.v4.new_code_cell("""
# Évaluation du modèle
evaluate_model(model, X_test, y_test)
"""),
    nbf.v4.new_code_cell("""
# Sauvegarde du modèle
save_model(model, 'models/final_model.pkl')
print('Modèle sauvegardé dans le fichier models/final_model.pkl.')
"""),
]

# 3. Ajout de la partie pour les prédictions
prediction_cells = [
    nbf.v4.new_code_cell("""
# Chargement des données de test
print("\nChargement des données de test...")
test_data = pd.read_csv('data/raw/test.csv', encoding='utf-8')
print('Aperçu des données de test:')
print(test_data.head())
"""),
    nbf.v4.new_code_cell("""
# Nettoyage des données de test
cleaned_test_data = clean_data(test_data)
print('Données de test nettoyées:')
print(cleaned_test_data.head())
"""),
    nbf.v4.new_code_cell("""
# Préparation des données de test pour la prédiction
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder

# Identifier les colonnes
categorical_features = cleaned_test_data.select_dtypes(include=['category', 'object']).columns
numeric_features = cleaned_test_data.select_dtypes(include=['int64', 'float64']).columns

# Transformer les données avec le même préprocesseur
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)
X_test_final = preprocessor.fit_transform(cleaned_test_data)
"""),
    nbf.v4.new_code_cell("""
# Chargement du modèle sauvegardé
print("\nChargement du modèle sauvegardé...")
model = load_model('models/final_model.pkl')
"""),
    nbf.v4.new_code_cell("""
# Réalisation des prédictions
predictions = model.predict(X_test_final)
cleaned_test_data['Survived'] = predictions

# Aperçu des prédictions
print("\nPrédictions sur les données de test:")
print(cleaned_test_data[['Survived']].head())

# Sauvegarde des résultats
output_path = 'data/processed/predictions.csv'
cleaned_test_data[['PassengerId', 'Survived']].to_csv(output_path, index=False)
print(f"Prédictions sauvegardées dans {output_path}")
"""),
]

# Ajouter toutes les cellules dans le notebook
nb['cells'] = eda_cells + pipeline_cells + prediction_cells

# 4. Sauvegarder le notebook dans un fichier
output_path = "titanic_eda_demo.ipynb"
with open(output_path, 'w', encoding='utf-8') as f:
    nbf.write(nb, f)

print(f"Notebook généré avec succès : {output_path}")


Notebook généré avec succès : titanic_eda_demo.ipynb
